In [ ]:
import sqlite3
import tkinter as tk
from tkinter import ttk, messagebox
import pandas as pd


class CabinAssignmentGUI:
    def __init__(self, db_file="lusitania_many_to_many.db"):
        self.db_file = db_file
        self.conn = sqlite3.connect(self.db_file)
        self.conn.execute("PRAGMA foreign_keys = ON;")

        self.root = tk.Tk()
        self.root.title("Lusitania Many-to-Many Cabin Manager")
        self.root.geometry("950x550")

        self.create_widgets()
        self.load_passengers()
        self.load_cabins()
        self.refresh_assignments()

        self.root.protocol("WM_DELETE_WINDOW", self.close_app)
        self.root.mainloop()

    def create_widgets(self):
        title = tk.Label(
            self.root,
            text="Passenger Cabin Assignment Manager",
            font=("Arial", 16, "bold")
        )
        title.pack(pady=10)

        control_frame = tk.Frame(self.root)
        control_frame.pack(pady=10)

        # Passenger dropdown
        tk.Label(control_frame, text="Passenger:", font=("Arial", 11)).grid(
            row=0, column=0, padx=5, pady=5, sticky="e"
        )
        self.passenger_combo = ttk.Combobox(control_frame, width=35, state="readonly")
        self.passenger_combo.grid(row=0, column=1, padx=5, pady=5)

        # Cabin dropdown
        tk.Label(control_frame, text="New Cabin:", font=("Arial", 11)).grid(
            row=0, column=2, padx=5, pady=5, sticky="e"
        )
        self.cabin_combo = ttk.Combobox(control_frame, width=30, state="readonly")
        self.cabin_combo.grid(row=0, column=3, padx=5, pady=5)

        # Buttons
        move_button = tk.Button(
            control_frame,
            text="Move Passenger",
            width=15,
            command=self.move_passenger
        )
        move_button.grid(row=0, column=4, padx=10, pady=5)

        delete_passenger_button = tk.Button(
            control_frame,
            text="Delete Passenger",
            width=15,
            command=self.delete_passenger
        )
        delete_passenger_button.grid(row=1, column=1, padx=5, pady=10)

        delete_cabin_button = tk.Button(
            control_frame,
            text="Delete Cabin",
            width=15,
            command=self.delete_cabin
        )
        delete_cabin_button.grid(row=1, column=3, padx=5, pady=10)

        refresh_button = tk.Button(
            control_frame,
            text="Refresh View",
            width=15,
            command=self.refresh_all
        )
        refresh_button.grid(row=1, column=4, padx=10, pady=10)

        # Info label
        self.info_label = tk.Label(
            self.root,
            text="Assignments will appear below",
            font=("Arial", 11),
            fg="blue"
        )
        self.info_label.pack(pady=5)

        # Treeview
        columns = (
            "assignment_id",
            "passenger_id",
            "personal_name",
            "family_name",
            "cabin_id",
            "cabin_class",
            "ship_level"
        )

        self.tree = ttk.Treeview(self.root, columns=columns, show="headings", height=18)

        self.tree.heading("assignment_id", text="Assignment ID")
        self.tree.heading("passenger_id", text="Passenger ID")
        self.tree.heading("personal_name", text="First Name")
        self.tree.heading("family_name", text="Family Name")
        self.tree.heading("cabin_id", text="Cabin ID")
        self.tree.heading("cabin_class", text="Cabin Class")
        self.tree.heading("ship_level", text="Deck")

        self.tree.column("assignment_id", width=95, anchor="center")
        self.tree.column("passenger_id", width=90, anchor="center")
        self.tree.column("personal_name", width=120, anchor="center")
        self.tree.column("family_name", width=120, anchor="center")
        self.tree.column("cabin_id", width=80, anchor="center")
        self.tree.column("cabin_class", width=100, anchor="center")
        self.tree.column("ship_level", width=100, anchor="center")

        self.tree.pack(fill="both", expand=True, padx=15, pady=10)

        scrollbar = ttk.Scrollbar(self.root, orient="vertical", command=self.tree.yview)
        self.tree.configure(yscrollcommand=scrollbar.set)
        #scrollbar.place(x=915, y=145, height=365)

    def load_passengers(self):
        query = """ SELECT passenger_id, family_name, personal_name FROM passenger """
        df = pd.read_sql(query, self.conn)
        self.passenger_map = {}
        display_values = []

        for _, row in df.iterrows():
            display_text = f"{row['passenger_id']} - {row['family_name']}, {row['personal_name']}"
            display_values.append(display_text)
            self.passenger_map[display_text] = row["passenger_id"]

        self.passenger_combo["values"] = display_values
        if display_values:
            self.passenger_combo.current(0)

    def load_cabins(self):
        query = """ SELECT cabin_id, cabin_class, ship_level FROM cabin_inventory """
        df = pd.read_sql(query, self.conn)
        self.cabin_map = {}
        display_values = []

        for _, row in df.iterrows():
            display_text = f"{row['cabin_id']} - {row['cabin_class']} - {row['ship_level']}"
            display_values.append(display_text)
            self.cabin_map[display_text] = row["cabin_id"]

        self.cabin_combo["values"] = display_values
        if display_values:
            self.cabin_combo.current(0)

    def refresh_assignments(self):
        query = """ SELECT
                a.assignment_id,
                p.passenger_id,
                p.personal_name,
                p.family_name,
                c.cabin_id,
                c.cabin_class,
                c.ship_level
            FROM cabin_assignment a
            JOIN passenger p
            ON a.passenger_id = p.passenger_id
            JOIN cabin_inventory c
            ON a.cabin_id = c.cabin_id
            ORDER BY c.cabin_id, p.family_name, p.personal_name
            """
        df = pd.read_sql(query, self.conn)

        for item in self.tree.get_children():
            self.tree.delete(item)

        for _, row in df.iterrows():
            self.tree.insert(
                "",
                "end",
                values=(
                    row["assignment_id"],
                    row["passenger_id"],
                    row["personal_name"],
                    row["family_name"],
                    row["cabin_id"],
                    row["cabin_class"],
                    row["ship_level"]
                )
            )

        self.info_label.config(text=f"Showing {len(df)} current cabin assignments")

    def move_passenger(self):
        selected_passenger = self.passenger_combo.get()
        selected_cabin = self.cabin_combo.get()

        if not selected_passenger or not selected_cabin:
            messagebox.showwarning("Missing Selection", "Please select both a passenger and a cabin.")
            return

        passenger_id = self.passenger_map[selected_passenger]
        new_cabin_id = self.cabin_map[selected_cabin]

        self.conn.execute(""" UPDATE cabin_assignment SET cabin_id = ? WHERE passenger_id = ? """, (new_cabin_id, passenger_id))
        self.conn.commit()

        messagebox.showinfo("Success", "Passenger moved successfully.")
        self.refresh_assignments()

    def delete_passenger(self):
        selected_passenger = self.passenger_combo.get()

        if not selected_passenger:
            messagebox.showwarning("Missing Selection", "Please select a passenger.")
            return

        passenger_id = self.passenger_map[selected_passenger]

        confirm = messagebox.askyesno(
            "Confirm Delete",
            "Delete this passenger?\nRelated assignments will be deleted automatically."
        )
        if not confirm:
            return

        # No JOIN needed here
        self.conn.execute("""
        DELETE FROM passenger
        WHERE passenger_id = ?
        """, (passenger_id,))
        self.conn.commit()

        messagebox.showinfo("Deleted", "Passenger deleted.")
        self.refresh_all()

    def delete_cabin(self):
        selected_cabin = self.cabin_combo.get()

        if not selected_cabin:
            messagebox.showwarning("Missing Selection", "Please select a cabin.")
            return

        cabin_id = self.cabin_map[selected_cabin]

        confirm = messagebox.askyesno(
            "Confirm Delete",
            "Delete this cabin?\nRelated assignments will be deleted automatically."
        )
        if not confirm:
            return

        # No JOIN needed here
        self.conn.execute("""
        DELETE FROM cabin_inventory
        WHERE cabin_id = ?
        """, (cabin_id,))
        self.conn.commit()

        messagebox.showinfo("Deleted", "Cabin deleted.")
        self.refresh_all()

    def refresh_all(self):
        self.load_passengers()
        self.load_cabins()
        self.refresh_assignments()

    def close_app(self):
        self.conn.close()
        self.root.destroy()


if __name__ == "__main__":
    CabinAssignmentGUI("lusitania_many_to_many.db")